# ステージ0–1: 環境確認とリファレンスQC

oligo 5株 metatranscriptome の準備段階。**シーケンスを始める前に走らせる。**

| 株ID | 株 | 注意 |
|---|---|---|
| `Bth_VPI` | *B. thetaiotaomicron* VPI-5482 | プラスミド p5482 を含めること |
| `Bth_St8` | *B. thetaiotaomicron* CE-St8 | **Bth_VPI と同種・近縁** |
| `Bun_8492` | *B. uniformis* ATCC 8492 | 別種。問題なし |
| `Odo_St23` | *Odoribacter* CE-St23 | **Odo_St24 とほぼ同一ゲノム** |
| `Odo_St24` | *Odoribacter* CE-St24 | **Odo_St23 とほぼ同一ゲノム** |

このnotebookが答える問い: **「どの遺伝子なら、どの株由来か正しく判定できるか」**

5株中2ペアが近縁なので、リードが複数株に等しくマップされて株の区別がつかない遺伝子が
相当数ある。それが何%なのかを、実験を始める前に数値で確定させる。

最後にゲート判定の表が出る。**そこで一度止まって判断する。**

## セル1: 設定
ここだけ環境に合わせて書き換える。以降のセルは触らない。

In [ ]:
from pathlib import Path

# ── プロジェクトのルート（このnotebookを置く場所）
ROOT = Path("/mnt/hdd1/tanoue/oligo5_metatx")   # ★ 実際のパスに変更

# ── 入力ゲノム。ファイル名が株IDになる（例: Bth_VPI.fna → 株ID "Bth_VPI"）
GENOME_DIR = ROOT / "genomes"

# ── 実際のシーケンスのリード長（片側）。mappability の結果がこの値で変わる
READLEN = 150                                    # ★ 実測値に変更

# ── 擬似リードを刻む間隔。小さいほど精密だが遅い。10で十分
STEP = 10

# ── 使用スレッド数
THREADS = 16

# ── 近縁ペアとみなす組み合わせ（ゲート判定で重点的に見る）
FOCUS_PAIRS = [("Bth_VPI", "Bth_St8"), ("Odo_St23", "Odo_St24")]

# ── 出力先（自動作成）
ANI_DIR  = ROOT / "ani"
PAN_DIR  = ROOT / "pangenome"
MAP_DIR  = ROOT / "mappability"
REF_DIR  = ROOT / "ref"
LOG_DIR  = ROOT / "logs"
BAKTA_DIR = ROOT / "bakta"        # bakta/<株ID>/<株ID>.ffn, .gff3 を想定

for d in (ANI_DIR, PAN_DIR, MAP_DIR, REF_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT     :", ROOT)
print("READLEN  :", READLEN, "bp")
print("THREADS  :", THREADS)

## セル2: ヘルパー関数
外部コマンドを呼んで、標準出力とエラーを見やすく表示するだけ。

In [ ]:
import subprocess, shutil, time, textwrap
import pandas as pd
from collections import defaultdict, Counter

def sh(cmd, log=None, quiet=False):
    """シェルコマンドを実行する。log にパスを渡すと stderr をそこへ保存。
    戻り値は標準出力の文字列。失敗したら例外ではなく内容を表示して None を返す。"""
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    el = time.time() - t0
    if log:
        Path(log).write_text(r.stderr)
    if r.returncode != 0:
        print(f"[失敗 rc={r.returncode}] {cmd}")
        print(textwrap.indent(r.stderr[-2000:], "  "))
        return None
    if not quiet:
        print(f"[完了 {el:.1f}s] {cmd.split()[0]}")
    return r.stdout

def have(tool):
    """コマンドが PATH にあるか"""
    return shutil.which(tool) is not None

def read_fasta(path):
    """FASTA を (名前, 配列) のジェネレータで返す。メモリに全部載せない。"""
    name, seq = None, []
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                if name: yield name, "".join(seq)
                name, seq = line[1:].strip(), []
            else:
                seq.append(line.strip())
    if name: yield name, "".join(seq)

print("ヘルパー関数を定義しました")

---
# ステージ0: 環境確認

## セル3: ツールの有無を確認
足りないものがあれば、次のセルが導入コマンドを出す。

In [ ]:
TOOLS = ["fastANI", "minimap2", "bowtie2", "bowtie2-build", "samtools",
         "panaroo", "checkm2", "bakta", "emapper.py",
         "fastqc", "multiqc", "fastp", "salmon", "seqkit"]

missing = []
print(f"{'ツール':<16}{'場所'}")
print("-" * 70)
for t in TOOLS:
    p = shutil.which(t)
    print(f"{t:<16}{p if p else '— 未インストール'}")
    if not p: missing.append(t)

print()
if missing:
    print("不足:", ", ".join(missing))
else:
    print("すべて揃っています")

## セル4: 不足分の導入コマンドを表示
出力をコピーしてターミナルで実行する（notebookからは実行しない）。

In [ ]:
# conda パッケージ名はコマンド名と違うものがあるので対応表を持つ
CONDA_PKG = {"fastANI": "fastani", "bowtie2-build": "bowtie2",
             "emapper.py": "eggnog-mapper", "samtools": "samtools"}

if missing:
    pkgs = sorted({CONDA_PKG.get(t, t) for t in missing})
    print("# ターミナルで実行:")
    print("mamba install -y -c conda-forge -c bioconda \\")
    print("  " + " ".join(pkgs))
    print()
    print("# featureCounts は subread パッケージに含まれる")
    print("# （ステージ3で使う。今は不要）")
else:
    print("導入するものはありません")

## セル5: Rパッケージの確認
ステージ4〜5（DESeq2, fgsea）で使う。今の段階では無くても先に進める。

In [ ]:
RPKG = ["DESeq2", "Rsubread", "apeglm", "ashr", "fgsea",
        "clusterProfiler", "tidyverse", "pheatmap", "data.table"]

if have("Rscript"):
    code = 'for(p in c(%s)) cat(sprintf("%%-18s %%s\\n", p, ifelse(requireNamespace(p,quietly=TRUE), as.character(packageVersion(p)), "— 未インストール")))' % \
           ",".join(f'"{p}"' for p in RPKG)
    out = sh(f"Rscript -e '{code}'", quiet=True)
    print(out or "Rscript の実行に失敗しました")
else:
    print("Rscript が見つかりません（ステージ4までに必要）")

## セル6: 入力ゲノムの確認
5本そろっているか、サイズは妥当か。株IDはファイル名から取る。

In [ ]:
import re

genomes = {}          # 株ID -> Path
for ext in ("*.fna", "*.fasta", "*.fa"):
    for f in sorted(GENOME_DIR.glob(ext)):
        genomes[f.stem] = f

if not genomes:
    print(f"!! {GENOME_DIR} にゲノムがありません")
else:
    print(f"{'株ID':<14}{'contig数':>9}{'全長 Mb':>10}{'最長 contig Mb':>16}")
    print("-" * 52)
    glen = {}
    for sid, f in genomes.items():
        lens = [len(s) for _, s in read_fasta(f)]
        glen[sid] = sum(lens)
        print(f"{sid:<14}{len(lens):>9}{sum(lens)/1e6:>10.2f}{max(lens)/1e6:>16.2f}")
    print()
    print(f"合計 {len(genomes)} 株")
    # 株IDが FOCUS_PAIRS と合っているか確認
    for a, b in FOCUS_PAIRS:
        for x in (a, b):
            if x not in genomes:
                print(f"!! FOCUS_PAIRS の '{x}' が genomes/ にありません（ファイル名を確認）")

---
# ステージ1: リファレンスQC

ここからが本題。4つの問いに順番に答える。

1. **ANI** — 株どうしはどれくらい似ているか
2. **一意配列量** — 一方にしかない配列はどれだけあるか（kb）
3. **pangenome** — 一方にしかない遺伝子は何個か
4. **mappability** — 実リード長で、どのCDSが正しい株に割り当てられるか ← **これが最終判定**

1・2 は Bakta 不要なのですぐ走る。3・4 は Bakta 出力が必要。

## セル7: fastANI で全ペアのANIを計算

ANI = Average Nucleotide Identity（平均塩基一致度）。
`--ql` と `--rl` に同じリストを渡すと総当たりになる。5株なので25通り、数分で終わる。

In [ ]:
genome_list = ANI_DIR / "genome_list.txt"
genome_list.write_text("\n".join(str(f) for f in genomes.values()) + "\n")
ani_out = ANI_DIR / "ani_all.tsv"

if not have("fastANI"):
    print("!! fastANI がありません。セル4の導入コマンドを実行してから戻ってください")
elif ani_out.exists() and ani_out.stat().st_size > 0:
    print("既に結果があります:", ani_out, "（作り直すなら削除してから再実行）")
else:
    sh(f"fastANI --ql {genome_list} --rl {genome_list} -o {ani_out} -t {THREADS}",
       log=LOG_DIR / "fastani.log")

print("出力:", ani_out, "/ 存在:", ani_out.exists())

## セル8: ANI の結果を表示

**注意: 3列目のANIだけ見ても判断できない。**
ANI は「アラインできた領域の一致度」なので、ANI 99.9% でもアラインされたのが
ゲノムの70%だけなら、残り30%は相手に存在しない配列ということになる。

fastANI の 4列目 / 5列目 = アラインされたフラグメントの割合。**こちらが本題。**

In [ ]:
if not (ani_out.exists() and ani_out.stat().st_size > 0):
    raise SystemExit("ani_all.tsv がありません。セル7を先に成功させてください")

ani = pd.read_csv(ani_out, sep="\t", header=None,
                  names=["q_path", "r_path", "ANI", "n_frag_match", "n_frag_total"])
ani["query"] = ani.q_path.map(lambda p: Path(p).stem)
ani["ref"]   = ani.r_path.map(lambda p: Path(p).stem)
ani["aligned_pct"] = 100 * ani.n_frag_match / ani.n_frag_total

tbl = (ani[ani["query"] != ani["ref"]]
         .sort_values("ANI", ascending=False)
         [["query", "ref", "ANI", "aligned_pct"]]
         .reset_index(drop=True))

pd.set_option("display.width", 140)
print(tbl.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

print()
print("目安:  ANI >= 95%  → 同種")
print("       ANI >= 99%  → 非常に近縁。コア遺伝子での株の区別は困難")
print("       aligned_pct が低いほど、一方にしかない配列が多い（= 区別できる余地がある）")

## セル9: minimap2 で全ペアの一意配列量を計算

ANI では分からない「**相手にアラインしない領域が何 kb あるか**」を測る。
これがゲート判定の主役の数字。

`-x asm5` は「ほぼ同一のアセンブリ同士を比べる」プリセット。

In [ ]:
if not have("minimap2"):
    print("minimap2 がありません。セル4の導入コマンドを実行してください")
else:
    n = 0
    for qa, fa in genomes.items():
        for qb, fb in genomes.items():
            if qa == qb: continue
            paf = ANI_DIR / f"{qa}_vs_{qb}.paf"
            if paf.exists() and paf.stat().st_size > 0:
                continue                      # 既にあればスキップ（やり直しが速い）
            sh(f"minimap2 -x asm5 -t {THREADS} {fb} {fa} > {paf}",
               log=LOG_DIR / "minimap2.log", quiet=True)
            n += 1
    print(f"{n} ペアを新規に計算（既存分はスキップ）")
    print("PAF:", len(list(ANI_DIR.glob('*_vs_*.paf'))), "ファイル")

## セル10: 一意配列量を集計

**ここが落とし穴。** PAF には同じ領域に重なる複数のアラインメントが出るので、
単純に長さを足すと被覆を過大評価し、結果として**一意配列を過小評価**する。
（検証例: 正解 700 bp のところを 800 bp と出す）

そのため、contig ごとに区間をソートしてマージしてから合計する。

In [ ]:
def covered_bp(paf_path):
    """PAF から「query のどれだけが reference にアラインしたか」を返す。
    区間をマージしてから合計するので、重複アラインメントを二重計上しない。"""
    iv, ln = defaultdict(list), {}
    with open(paf_path) as f:
        for line in f:
            p = line.split("\t")
            if len(p) < 4: continue
            ln[p[0]] = int(p[1])                       # query contig 長
            iv[p[0]].append((int(p[2]), int(p[3])))    # アライン区間 [start, end)
    total = sum(ln.values())
    cov = 0
    for c, v in iv.items():
        v.sort()
        s, e = v[0]
        for a, b in v[1:]:
            if a <= e:            # 重なる → 伸ばす
                e = max(e, b)
            else:                 # 離れている → 確定して次へ
                cov += e - s
                s, e = a, b
        cov += e - s
    return total, cov

pafs = sorted(ANI_DIR.glob("*_vs_*.paf"))
if not pafs:
    raise SystemExit("PAF がありません。セル9を先に成功させてください")

rows = []
for paf in pafs:
    qa, qb = paf.stem.split("_vs_")
    total, cov = covered_bp(paf)
    if total == 0:
        continue
    rows.append({"query": qa, "reference": qb,
                 "全長_Mb": total / 1e6,
                 "被覆_pct": 100 * cov / total,
                 "一意_kb": (total - cov) / 1e3})

uniq = pd.DataFrame(rows).sort_values("一意_kb").reset_index(drop=True)
print(uniq.to_string(index=False, float_format=lambda x: f"{x:9.1f}"))

## セル11: 近縁ペアだけを抜き出す

`FOCUS_PAIRS` に指定した2ペアについて、両方向の一意配列量を並べる。

In [ ]:
if uniq.empty:
    raise SystemExit("一意配列量が計算されていません。セル9-10を先に実行してください")

print("近縁ペアの一意配列量（両方向）")
print("=" * 62)
for a, b in FOCUS_PAIRS:
    sub = uniq[((uniq["query"] == a) & (uniq.reference == b)) |
               ((uniq["query"] == b) & (uniq.reference == a))]
    if sub.empty:
        print(f"{a} ↔ {b}: PAF がありません"); continue
    print(f"\n[{a}  ↔  {b}]")
    for _, r in sub.iterrows():
        print(f"  {r['query']:<12} のうち {r['reference']:<12} に無い配列: "
              f"{r['一意_kb']:8.1f} kb   (被覆 {r['被覆_pct']:.1f}%)")
    mn = sub["一意_kb"].min()
    verdict = ("GO — 同一マウスで可"      if mn >= 100 else
               "条件付き — 論じられる遺伝子を明示" if mn >= 10 else
               "STOP — 片方を外すか群を分ける")
    print(f"  → 少ない方が {mn:.1f} kb : {verdict}")

---
## セル12: Bakta の出力があるか確認

ここから先（pangenome, mappability）は Bakta のアノテーションが要る。
無ければ次のセルがコマンドを出すので、ターミナルで実行してから戻ってくる。

In [ ]:
ffn  = sorted(BAKTA_DIR.glob("*/*.ffn"))
gff3 = sorted(BAKTA_DIR.glob("*/*.gff3"))
print(f"bakta/*/*.ffn  : {len(ffn)} 本")
for f in ffn:  print("   ", f)
print(f"bakta/*/*.gff3 : {len(gff3)} 本")

bakta_ready = len(ffn) == len(genomes) and len(gff3) == len(genomes)
print()
print("Bakta 出力はそろっています" if bakta_ready
      else f"!! 5株ぶん必要です（現在 ffn={len(ffn)}, gff3={len(gff3)}）")

## セル13: Bakta のコマンドを出力（未実施の場合）
時間がかかるのでターミナルで実行する。

In [ ]:
if not bakta_ready:
    print("# ターミナルで実行（1株あたり数分〜十数分）")
    print(f"cd {ROOT}")
    for sid, f in genomes.items():
        print(f"bakta --db $BAKTA_DB --output bakta/{sid} --prefix {sid} \\")
        print(f"      --locus-tag {sid} --threads {THREADS} --force {f}")
    print()
    print("# --locus-tag に株IDを指定しておくと、遺伝子IDから株が分かって後が楽")
else:
    print("実行済みです。次へ進んでください")

---
## セル14: Panaroo で pangenome を作る

「どの遺伝子がどの株にあるか」の有無行列を作る。
**1株にしかない遺伝子 = 一意にマップできる遺伝子**なので、株差を論じられる候補になる。

In [ ]:
pa_csv = PAN_DIR / "gene_presence_absence.csv"

if not bakta_ready:
    print("Bakta 出力が揃っていないのでスキップ")
elif not have("panaroo"):
    print("panaroo がありません。セル4の導入コマンドを実行してください")
elif pa_csv.exists():
    print("既に結果があります:", pa_csv)
else:
    gffs = " ".join(str(f) for f in gff3)
    sh(f"panaroo -i {gffs} -o {PAN_DIR} --clean-mode strict -t {THREADS}",
       log=LOG_DIR / "panaroo.log")

## セル15: pangenome の集計
「何株に存在するか」別の遺伝子数と、株ごとの固有遺伝子数。

In [ ]:
if pa_csv.exists():
    import csv
    rows = list(csv.reader(open(pa_csv)))
    hdr, strains = rows[0], rows[0][3:]
    cnt_by_n = defaultdict(int)
    uniq_by_strain = defaultdict(int)

    for r in rows[1:]:
        present = [s for s, v in zip(strains, r[3:]) if v.strip()]
        cnt_by_n[len(present)] += 1
        if len(present) == 1:
            uniq_by_strain[present[0]] += 1

    print("何株に存在するか別の遺伝子数")
    print("-" * 36)
    for k in sorted(cnt_by_n):
        label = "全株に共通（コア）" if k == len(strains) else f"{k}株のみ"
        print(f"  {k}株に存在: {cnt_by_n[k]:6d}   {label}")

    print()
    print("株ごとの固有遺伝子数（その株にしかない遺伝子）")
    print("-" * 46)
    for s in strains:
        n = uniq_by_strain[s]
        flag = "" if n >= 70 else ("  ← 少ない" if n >= 10 else "  ← ほぼ無い")
        print(f"  {s:<28}{n:6d}{flag}")
else:
    print("pangenome の結果がまだありません")

---
## セル16: 統合リファレンスを作る

**重要: 配列を端どうしで繋いで1本にしてはいけない。** ファイルを連結して
1つの multi-FASTA にし、contig は独立したレコードのまま保つ。

**もう1つ重要: Bakta は contig を `contig_1`, `contig_2` … に改名する。**
そのまま5株分つなぐと名前が衝突するので、contig名の先頭に株IDを付ける
（`Bth_VPI~contig_1` の形）。後で「このリードはどの株に行ったか」を見るのにも使う。

In [ ]:
ref_fna = REF_DIR / "oligo5_all.fna"
idx     = REF_DIR / "oligo5_all"     # bowtie2 index の prefix（セル18・20で使う）

with open(ref_fna, "w") as out:
    ncontig = 0
    for sid, f in genomes.items():
        for name, seq in read_fasta(f):
            cid = name.split()[0]                  # 説明文を落として contig 名だけ
            out.write(f">{sid}~{cid}\n")
            for i in range(0, len(seq), 60):       # 60文字で折り返す
                out.write(seq[i:i+60] + "\n")
            ncontig += 1

print(f"統合リファレンス: {ref_fna}  ({ncontig} contig)")
print(f"サイズ: {ref_fna.stat().st_size/1e6:.1f} MB")

## セル17: contig名の確認

株IDが全部の contig に付いているか、名前の重複がないかを確認する。

In [ ]:
names = [n for n, _ in read_fasta(ref_fna)]
print(f"contig 総数: {len(names)}  /  ユニーク名: {len(set(names))}")
if len(names) != len(set(names)):
    print("!! 名前が重複しています")

print()
print("株ごとの contig 数:")
cc = defaultdict(int)
for n in names:
    cc[n.split("~")[0]] += 1
for sid in genomes:
    print(f"  {sid:<14}{cc[sid]:>5}")

print()
print("先頭5件:")
for n in names[:5]:
    print("  >" + n)

## セル18: bowtie2 index を作る
数分かかる。既にあればスキップ。

In [ ]:
if Path(str(idx) + ".1.bt2").exists():
    print("index は既にあります:", idx)
elif not have("bowtie2-build"):
    print("bowtie2-build がありません")
else:
    sh(f"bowtie2-build --threads {THREADS} {ref_fna} {idx}",
       log=LOG_DIR / "bt2build.log")

---
## セル19: 擬似リードを作る

各CDSの配列を、実際のリード長で刻んで FASTA にする。
リード名に `株ID~遺伝子ID~位置` を埋め込んでおき、
マップし戻したときに「本来どこから来たか」が分かるようにする。

`READLEN` はセル1で設定した値。**ここが実際のリード長と違うと結果が変わる。**

In [ ]:
sim_fa = MAP_DIR / "sim_reads.fa"

if not bakta_ready:
    print("Bakta 出力が揃っていないのでスキップ")
else:
    n_read, n_cds = 0, 0
    with open(sim_fa, "w") as out:
        for f in ffn:
            sid = f.parent.name                       # bakta/<株ID>/ の <株ID>
            for name, seq in read_fasta(f):
                gid = name.split()[0]
                n_cds += 1
                # CDS がリード長より短い場合も1本は出す
                for i in range(0, max(1, len(seq) - READLEN + 1), STEP):
                    out.write(f">{sid}~{gid}~{i}\n{seq[i:i+READLEN]}\n")
                    n_read += 1
    print(f"CDS {n_cds:,} 本から 擬似リード {n_read:,} 本")
    print(f"（長さ {READLEN} bp、{STEP} bp 刻み）")
    print("出力:", sim_fa)

## セル20: 擬似リードを統合リファレンスに戻してマップ

自分の出身株に一意にマップできれば「区別できるCDS」。
別の株にも同じくらいよくマップされると bowtie2 は MAPQ を下げるので、
**MAPQ >= 10 を「一意」とみなす。**

同時に、一意でなかったリードがどの株へ行ったか（= 混同相手）も記録する。

In [ ]:
mapq_tsv = MAP_DIR / "cds_mappability.tsv"

if not (bakta_ready and sim_fa.exists()):
    print("擬似リードがないのでスキップ")
elif not Path(str(idx) + ".1.bt2").exists():
    print("bowtie2 index がありません。セル18を先に実行してください")
else:
    import collections
    tot    = collections.Counter()      # 遺伝子 -> 擬似リード総数
    uniq_c = collections.Counter()      # 遺伝子 -> 一意マップ数
    conf   = collections.defaultdict(collections.Counter)  # 遺伝子 -> 混同相手の株
    src_of = {}

    cmd = (f"bowtie2 -f -x {idx} -U {sim_fa} -p {THREADS} --no-unal "
           f"2> {LOG_DIR/'bt2_mappability.log'} | samtools view -")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, text=True, bufsize=1<<20)

    for line in proc.stdout:
        p = line.split("\t", 6)
        if len(p) < 5: continue
        src, gid, _ = p[0].split("~", 2)          # 出身: 株ID, 遺伝子ID
        dst = p[2].split("~", 1)[0]               # 行き先の contig が属する株
        key = f"{src}~{gid}"
        src_of[key] = src
        tot[key] += 1
        if int(p[4]) >= 10:
            uniq_c[key] += 1
        elif dst != src:
            conf[key][dst] += 1
    proc.wait()

    with open(mapq_tsv, "w") as out:
        out.write("strain\tgene\tn_sim\tn_unique\tmappability\ttop_confused_with\n")
        for k, t in tot.items():
            best = conf[k].most_common(1)
            out.write(f"{src_of[k]}\t{k}\t{t}\t{uniq_c[k]}\t{uniq_c[k]/t:.3f}\t"
                      f"{best[0][0] if best else '-'}\n")
    print(f"{len(tot):,} CDS を集計 → {mapq_tsv}")

## セル21: 株ごとの mappability サマリー

**これがこのnotebookの最終出力。**

- `mappability` が 1.0 に近い = その遺伝子は株を正しく判定できる
- 0 に近い = どの株由来か分からない

In [ ]:
if mapq_tsv.exists():
    mp = pd.read_csv(mapq_tsv, sep="\t")
    s = (mp.groupby("strain")
           .agg(CDS数=("gene", "size"),
                平均mappability=("mappability", "mean"),
                中央値=("mappability", "median"),
                区別困難=("mappability", lambda x: (x < 0.5).sum()),
                ほぼ不能=("mappability", lambda x: (x < 0.1).sum()))
           .reset_index())
    s["区別困難_pct"] = 100 * s["区別困難"] / s["CDS数"]
    s["ほぼ不能_pct"] = 100 * s["ほぼ不能"] / s["CDS数"]
    print(s.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
else:
    print("mappability の結果がまだありません")

## セル22: どの株どうしが混同されているか
区別困難なCDSが、実際にどの株と取り違えられているか。

In [ ]:
if mapq_tsv.exists():
    bad = mp[(mp.mappability < 0.5) & (mp.top_confused_with != "-")]
    pair = (bad.groupby(["strain", "top_confused_with"])
               .size().reset_index(name="CDS数")
               .sort_values("CDS数", ascending=False))
    print("mappability < 0.5 のCDSの混同相手")
    print("-" * 56)
    for _, r in pair.iterrows():
        print(f"  {r.strain:<14} ↔ {r.top_confused_with:<14} {r['CDS数']:6d} CDS")

## セル23: mappability の分布を図にする
株ごとのヒストグラム。二峰性（0付近と1付近に山）になるのが典型。

In [ ]:
if mapq_tsv.exists():
    import matplotlib.pyplot as plt
    strains_ = sorted(mp.strain.unique())
    fig, axes = plt.subplots(1, len(strains_), figsize=(3.1*len(strains_), 2.9), sharey=True)
    if len(strains_) == 1: axes = [axes]
    for ax, sid in zip(axes, strains_):
        v = mp.loc[mp.strain == sid, "mappability"]
        ax.hist(v, bins=25, range=(0, 1), color="#2a78d6", edgecolor="white", linewidth=.4)
        ax.axvline(0.5, color="#d03b3b", lw=1, ls="--")
        ax.set_title(f"{sid}\n平均 {v.mean():.3f}", fontsize=9)
        ax.set_xlabel("mappability")
    axes[0].set_ylabel("CDS数")
    fig.suptitle(f"CDSごとの株判別可能性（リード長 {READLEN} bp、赤線 = 0.5）", fontsize=10)
    fig.tight_layout()
    plt.savefig(MAP_DIR / "mappability_hist.pdf")
    plt.show()

---
# ゲート1: 判定

ここまでの数字をまとめる。**この先に進んでよいかを、ここで人が判断する。**

In [ ]:
print("=" * 72)
print("ゲート1  近縁ペアを同じマウスに入れてよいか")
print("=" * 72)

for a, b in FOCUS_PAIRS:
    print(f"\n■ {a}  ↔  {b}")

    # ANI
    row = ani[(ani["query"] == a) & (ani["ref"] == b)]
    if not row.empty:
        print(f"   ANI            : {row.ANI.iloc[0]:.3f} %  "
              f"(アライン率 {row.aligned_pct.iloc[0]:.1f}%)")

    # 一意配列
    sub = uniq[((uniq["query"] == a) & (uniq.reference == b)) |
               ((uniq["query"] == b) & (uniq.reference == a))]
    u_kb = sub["一意_kb"].min() if not sub.empty else None
    if u_kb is not None:
        print(f"   一意配列（少ない方）: {u_kb:,.0f} kb")

    # 固有遺伝子
    if pa_csv.exists():
        ua, ub = uniq_by_strain.get(a, 0), uniq_by_strain.get(b, 0)
        print(f"   固有遺伝子     : {a} {ua} 個 / {b} {ub} 個  （少ない方 {min(ua,ub)}）")
    else:
        ua = ub = None

    # mappability
    if mapq_tsv.exists():
        for x in (a, b):
            v = mp.loc[mp.strain == x, "mappability"]
            if len(v):
                print(f"   {x:<12} : 区別可能なCDS {(v>=0.5).sum():,} / {len(v):,} "
                      f"({100*(v>=0.5).mean():.1f}%)")

    # 判定
    if u_kb is None:
        verdict = "（データ不足）"
    elif u_kb >= 100 and (ua is None or min(ua, ub) >= 70):
        verdict = "GO — 同一マウスで可。アクセサリ遺伝子で株差を論じる"
    elif u_kb >= 10:
        verdict = "条件付き — 論じられる遺伝子を明示して進む。コアは種レベルで合算"
    else:
        verdict = "STOP — 同一マウスに入れる意味がない。片方を外すか群を分ける"
    print(f"   判定           : {verdict}")

print()
print("=" * 72)
print("この結果を確認してから、ステージ2（リファレンス構築）へ進むこと。")
print("STOP が出たペアがある場合は、実験デザインの見直しを先に行う。")

---
## 次にやること

| ゲート1の結果 | 次 |
|---|---|
| 全ペア GO | ステージ2（リファレンス構築）へ。`mappability` 列を GTF に持たせる |
| 条件付き | 同上。ただし解析対象CDSを `mappability >= 0.5` に限定する方針を決めておく |
| STOP あり | **実験デザインの見直しを先に。** 片方を外す／群を分ける |

`mappability/cds_mappability.tsv` はこの先ずっと使う。
株レベルの所見を出すたびに、その遺伝子が区別可能だったかを必ず確認する。